# node21det: fila de execuções no Colab

Roda várias execuções em sequência: acabando uma, começa a próxima.
Ambiente: **GPU** (Ambiente de execução → Alterar o tipo → T4 ou L4).

Preencha a `FILA` e rode tudo (Ambiente de execução → Executar tudo). Se a sessão
cair, abra de novo e rode tudo: execuções completas são puladas e a interrompida
retoma do `last.pt` salvo no Drive.

## 1. Configuração

In [ ]:
# Pasta do Drive com images/ e metadata.csv
DRIVE_DATA = "/content/drive/MyDrive/dataset_node21/cxr_images/proccessed_data"
# Onde ficam checkpoints e históricos (persistem entre sessões)
DRIVE_RUNS = "/content/drive/MyDrive/tcc/runs"

# (CONFIG, RUN_NAME, EXTRA_ARGS), na ordem em que devem rodar.
# Um RUN_NAME novo para cada execução: com um nome existente o treino retoma aquele last.pt.
FILA = [
    # Comparação de arquiteturas com sorteio balanceado (D38). A RetinaNet vai primeiro:
    # é a mais incerta, e se parar cedo a fila segue para a Faster R-CNN.
    ("configs/retinanet_balanced.yaml", "s4_retinanet_balanced", "--mode select"),
    ("configs/frcnn_balanced.yaml",     "s4_frcnn_balanced",     "--mode select"),
]

PARAR_NO_ERRO = False        # True: uma execução que falha interrompe a fila; False: segue para a próxima
DESCONECTAR_AO_FIM = True    # libera a GPU (e para de gastar unidades) quando a fila terminar

## 2. GPU e Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 3. Código

In [ ]:
import os
if os.path.exists("/content/node21det"):
    !git -C /content/node21det pull -q
else:
    !git clone -q https://github.com/SchafNunes/node21det.git /content/node21det
%cd /content/node21det
!git log --oneline -1
!pip install -q -e .

## 4. Dados no disco local da sessão

Na primeira vez copia `images/` do Drive arquivo por arquivo, com nova tentativa
em caso de erro de leitura do Drive, e grava `node21_images.tar` no Drive. Nas
próximas sessões copia só o `.tar`. Se for interrompida, rode de novo: continua
de onde parou.

In [ ]:
import hashlib, os, shutil, subprocess, time

LOCAL = "/content/data"
TAR = f"{DRIVE_DATA}/node21_images.tar"
SRC = f"{DRIVE_DATA}/images"
DST = f"{LOCAL}/images"
EXPECTED = 4882

print("conteúdo de DRIVE_DATA:", sorted(os.listdir(DRIVE_DATA)))
os.makedirs(DST, exist_ok=True)

def copy_with_retry(src, dst, tries=5):
    for k in range(tries):
        try:
            shutil.copyfile(src, dst)
            return None
        except OSError as e:
            time.sleep(2 * (k + 1))
            err = e
    return err

t0 = time.time()
have = len([f for f in os.listdir(DST) if f.endswith(".mha")])
if have < EXPECTED and os.path.exists(TAR):
    print("extraindo o .tar do Drive")
    subprocess.run(["tar", "-xf", TAR, "-C", LOCAL], check=True)
elif have < EXPECTED:
    names = sorted(f for f in os.listdir(SRC) if f.endswith(".mha"))
    print(f"{len(names)} .mha no Drive; {have} já copiados")
    failed = []
    for i, name in enumerate(names):
        src, dst = f"{SRC}/{name}", f"{DST}/{name}"
        if os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(src):
            continue
        err = copy_with_retry(src, dst)
        if err:
            failed.append((name, repr(err)))
        if i % 250 == 0:
            print(f"  {i}/{len(names)}  {time.time() - t0:.0f} s", flush=True)
    if failed:
        print(f"{len(failed)} falharam; rode a célula de novo. Exemplos: {failed[:3]}")
    else:
        print("cópia completa; gravando o .tar no Drive para as próximas sessões")
        subprocess.run(["tar", "-cf", TAR + ".tmp", "-C", LOCAL, "images"], check=True)
        os.replace(TAR + ".tmp", TAR)

shutil.copyfile(f"{DRIVE_DATA}/metadata.csv", f"{LOCAL}/metadata.csv")
n = len([f for f in os.listdir(DST) if f.endswith(".mha")])
md5 = hashlib.md5(open(f"{LOCAL}/metadata.csv", "rb").read()).hexdigest()
print(f"{n} imagens | metadata md5 {md5} | {time.time() - t0:.0f} s")
assert n == EXPECTED, "faltam imagens: rode a célula de novo"
assert md5 == "86c9d97611be7deeca6370cf4ef42503", "metadata.csv diferente do usado nas partições"

## 5. Fila

Cada execução grava a saída também em `DRIVE_RUNS/<RUN_NAME>/train.log`.
Uma execução conta como completa quando:

- `fold`: existe `fold_metrics.json`;
- `final`: `history.csv` tem as `--epochs` épocas;
- `select`: o `train.py` é chamado de novo e, se já terminou, sai logo depois de carregar o `last.pt`.

In [ ]:
import os, shlex, subprocess, sys, time
import pandas as pd

names = [r for _, r, _ in FILA]
assert len(names) == len(set(names)), "RUN_NAME repetido na FILA"
for cfg, _, _ in FILA:
    assert os.path.exists(cfg), f"config não encontrada: {cfg}"

def arg(extra, flag):
    toks = shlex.split(extra)
    return toks[toks.index(flag) + 1] if flag in toks else None

def complete(out, extra):
    mode = arg(extra, "--mode") or "select"
    if mode == "fold":
        return os.path.exists(f"{out}/fold_metrics.json")
    if mode == "final":
        h = f"{out}/history.csv"
        return os.path.exists(h) and len(pd.read_csv(h)) >= int(arg(extra, "--epochs"))
    return False

status = {}
for i, (cfg, run, extra) in enumerate(FILA, 1):
    out = f"{DRIVE_RUNS}/{run}"
    head = f"[{i}/{len(FILA)}] {run}  ({cfg} {extra})"
    if complete(out, extra):
        print(f"{head}: já completa, pulando")
        status[run] = "já completa"
        continue
    print(f"\n{'=' * 80}\n{head}: início {time.strftime('%H:%M:%S')}\n{'=' * 80}", flush=True)
    os.makedirs(out, exist_ok=True)
    cmd = [sys.executable, "-W", "ignore", "scripts/train.py", "--config", cfg, *shlex.split(extra),
           "--images", "/content/data/images", "--metadata", "/content/data/metadata.csv",
           "--splits", "splits/splits.csv", "--out", out]
    t0 = time.time()
    with open(f"{out}/train.log", "a") as log:
        log.write(f"\n### {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(cmd)}\n")
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        code = p.wait()
    minutes = (time.time() - t0) / 60
    status[run] = f"ok ({minutes:.0f} min)" if code == 0 else f"ERRO código {code} ({minutes:.0f} min)"
    print(f"{head}: {status[run]}", flush=True)
    if code != 0 and PARAR_NO_ERRO:
        break

print("\nResumo da fila:")
for run in names:
    print(f"  {run:30s} {status.get(run, 'não rodou')}")

## 6. Resultados

In [ ]:
import json
for _, run, extra in FILA:
    out = f"{DRIVE_RUNS}/{run}"
    print(f"\n## {run}")
    if os.path.exists(f"{out}/history.csv"):
        h = pd.read_csv(f"{out}/history.csv")
        cols = [c for c in ["epoch", "lr", "train_loss", "train_seconds", "gpu_peak_gb", "val_mean_sens",
                            "val_sens@0.25", "val_ap@0.2", "val_rank", "val_auc", "val_auc_outside_nodules"] if c in h]
        h["train_seconds"] = h["train_seconds"] / 60   # minutos
        display(h[cols].round(4))
    if os.path.exists(f"{out}/fold_metrics.json"):
        m = json.load(open(f"{out}/fold_metrics.json"))
        print({k: round(v, 4) for k, v in m.items() if isinstance(v, (int, float))})

In [ ]:
if DESCONECTAR_AO_FIM:
    from google.colab import runtime
    runtime.unassign()